# **Task 2: Feature Engineering & Preprocessing**

In [1]:
#importing necessary libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler , MinMaxScaler
import pandas as pd
import numpy as np
import joblib
import os

%matplotlib inline

In [3]:
tele_data=pd.read_csv("/content/Cleaned-Telco-Customer-Churn.csv")

In [4]:
tele_data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1 - 12
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25 - 36
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1 - 12
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,37 - 48
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1 - 12


In [5]:
tele_data.drop(columns=['tenure_group'], axis=1, inplace=True) # removing columns not required for processing
tele_data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## **1. Feature Engineering**


In [6]:
tele_data['Churn'] = tele_data['Churn'].map({'Yes': 1, 'No': 0})

1.1 Feature Engineering Average Monthly Spend

In [7]:
tele_data['AvgMonthlySpend'] = tele_data['MonthlyCharges'] /( tele_data['tenure']+1)

1.2 Feature Engineering Service Count

In [8]:
service_cols = ["PhoneService","OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]

tele_data["ServiceCount"] = 0

for col in service_cols:
    tele_data["ServiceCount"] += (tele_data[col].str.contains("Yes")).astype(int)

1.3 Feature Engineering Contract Value

In [9]:
tele_data["ContractValue"] = (tele_data["MonthlyCharges"] * tele_data["tenure"])

In [10]:
tele_data[["AvgMonthlySpend","ServiceCount","ContractValue",]].head()

,AvgMonthlySpend,ServiceCount,ContractValue
0,14.925000,1,29.85
1,1.627143,3,1936.30
2,17.950000,3,107.70
3,0.919565,3,1903.50
4,23.566667,1,141.40


## **2. Split data (70% train / 15% validation / 15% test)**

2.1 spliting features and target

In [11]:
X = tele_data.drop(['Churn'], axis=1)
y = tele_data['Churn']

2.2 First split

In [12]:
X_train, X_temp, y_train, y_temp = (train_test_split(X,y,test_size=0.30,stratify=y,random_state=42))

2.3 Second split

In [13]:
X_val, X_test, y_val, y_test = (train_test_split(X_temp,y_temp,test_size=0.50,stratify=y_temp,random_state=42))

2.4 verifying shapes

In [14]:
train_rows = X_train.shape[0]
val_rows = X_val.shape[0]
test_rows = X_test.shape[0]

total_rows = train_rows + val_rows + test_rows

print(f"Train set:      {train_rows} rows ({train_rows / total_rows:.1%})")
print(f"Validation set: {val_rows} rows ({val_rows / total_rows:.1%})")
print(f"Test set:       {test_rows} rows ({test_rows / total_rows:.1%})")


Train set:      4922 rows (70.0%)
Validation set: 1055 rows (15.0%)
Test set:       1055 rows (15.0%)


## **2. Encoding Categorical Variables**

3.1 converting target variable (Churn) into numeric format

3.2 Label encoding:

using label encoding on columns containing binary data because it will convert the data in the form of 0 and 1, and it keeps our dataset small and efficient, saves memory, and prevents our models from slowing down by completely avoiding the creation of unnecessary, redundant columns.

In [15]:
# label encoding binary valued columns
binary_cols = ["gender","Partner","Dependents","PhoneService","PaperlessBilling"]
le=LabelEncoder()
for col in binary_cols:
    X_train[col] = le.fit_transform(X_train[col])
    X_val[col] = le.transform(X_val[col])
    X_test[col] = le.transform(X_test[col])

In [16]:
tele_data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgMonthlySpend,ServiceCount,ContractValue
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,14.925000,1,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,One year,No,Mailed check,56.95,1889.50,0,1.627143,3,1936.30
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,17.950000,3,107.70
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,0.919565,3,1903.50
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,23.566667,1,141.40


3.3 One-Hot Encoding:

using one-hot encoding on columns containing multi-categorical values because it prevents the model from assuming a false mathematical order or ranking among unrelated categories, ensuring each category is treated as an independent, distinct feature without adding bias.

In [17]:
#using one-hot encoding
#multi_cols = ["MultipleLines","InternetService","OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies","Contract","PaymentMethod"]

X_train = pd.get_dummies(X_train, drop_first=True)
X_val = pd.get_dummies(X_val, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align columns to ensure train, val, and test have the exact same dummy layout
X_train, X_val = X_train.align(X_val, join='left', axis=1, fill_value=0)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [18]:
X_train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,AvgMonthlySpend,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
4491,0,0,0,1,12,1,1,78.30,909.25,6.023077,...,False,False,False,False,False,False,False,False,True,False
1928,1,0,0,0,20,1,1,19.70,415.90,0.938095,...,False,True,False,True,False,True,False,False,False,True
4660,0,0,0,1,2,1,1,61.20,125.95,20.400000,...,False,False,False,False,True,False,False,True,False,False
5672,0,1,1,0,34,1,0,64.20,2106.30,1.834286,...,True,False,True,False,False,True,False,False,False,False
3604,0,0,0,0,12,1,1,100.15,1164.30,7.703846,...,False,False,True,False,True,False,False,False,False,False


In [19]:
# Drop the 'tenure_group' column as it's non-numeric and not needed for modeling in its current form
if 'tenure_group' in tele_data.columns:
    tele_data = tele_data.drop('tenure_group', axis=1)
    print("Dropped 'tenure_group' column.")
else:
    print("'tenure_group' column not found.")


'tenure_group' column not found.


## **4. Feature scaling**

4.1 Applying StandardScaler

In [20]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
std_scaler = StandardScaler()

X_train_std = X_train.copy()
X_val_std = X_val.copy()
X_test_std = X_test.copy()

X_train_std[num_cols] = (std_scaler.fit_transform(X_train_std[num_cols]))
X_val_std[num_cols] = (std_scaler.transform(X_val_std[num_cols]))
X_test_std[num_cols] = (std_scaler.transform(X_test_std[num_cols]))

4.2 Applying MinMaxScaling

In [21]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
mm_scaler = MinMaxScaler()

X_train_mm = X_train.copy()
X_val_mm = X_val.copy()
X_test_mm = X_test.copy()

X_train_mm[num_cols] = (mm_scaler.fit_transform(X_train_mm[num_cols]))
X_val_mm[num_cols] = (mm_scaler.transform(X_val_mm[num_cols]))
X_test_mm[num_cols] = (mm_scaler.transform(X_test_mm[num_cols]))

4.3 Compare Scaling Methods

In [22]:
pd.DataFrame(X_train_std).describe()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,AvgMonthlySpend,ServiceCount,ContractValue
count,4922.000000,4922.000000,4922.000000,4922.000000,4.922000e+03,4922.000000,4922.000000,4.922000e+03,4.922000e+03,4922.000000,4922.000000,4922.000000
mean,0.501625,0.162536,0.485372,0.299269,9.960879e-17,0.903088,0.594880,1.999394e-16,3.175933e-17,5.727937,2.946973,2291.387058
std,0.500048,0.368979,0.499837,0.457984,1.000102e+00,0.295868,0.490965,1.000102e+00,1.000102e+00,8.653637,1.844997,2275.172529
min,0.000000,0.000000,0.000000,0.000000,-1.280909e+00,0.000000,0.000000,-1.535103e+00,-9.980191e-01,0.264384,0.000000,18.800000
25%,0.000000,0.000000,0.000000,0.000000,-9.554981e-01,1.000000,0.000000,-9.794060e-01,-8.304334e-01,1.254297,1.000000,396.862500
50%,1.000000,0.000000,0.000000,0.000000,-1.419706e-01,1.000000,1.000000,1.885914e-01,-3.933032e-01,2.070660,3.000000,1397.650000
75%,1.000000,0.000000,1.000000,1.000000,9.562914e-01,1.000000,1.000000,8.331178e-01,6.581690e-01,5.880515,4.000000,3793.112500
max,1.000000,1.000000,1.000000,1.000000,1.607113e+00,1.000000,1.000000,1.776770e+00,2.806797e+00,50.725000,7.000000,8510.400000


In [23]:
pd.DataFrame(X_train_mm).describe()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,AvgMonthlySpend,ServiceCount,ContractValue
count,4922.000000,4922.000000,4922.000000,4922.000000,4922.000000,4922.000000,4922.000000,4922.000000,4922.000000,4922.000000,4922.000000,4922.000000
mean,0.501625,0.162536,0.485372,0.299269,0.443525,0.903088,0.594880,0.463515,0.262304,5.727937,2.946973,2291.387058
std,0.500048,0.368979,0.499837,0.457984,0.346293,0.295868,0.490965,0.301975,0.262852,8.653637,1.844997,2275.172529
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.264384,0.000000,18.800000
25%,0.000000,0.000000,0.000000,0.000000,0.112676,1.000000,0.000000,0.167789,0.044046,1.254297,1.000000,396.862500
50%,1.000000,0.000000,0.000000,0.000000,0.394366,1.000000,1.000000,0.520459,0.158934,2.070660,3.000000,1397.650000
75%,1.000000,0.000000,1.000000,1.000000,0.774648,1.000000,1.000000,0.715070,0.435287,5.880515,4.000000,3793.112500
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,50.725000,7.000000,8510.400000


## **5. Handling class imbalance**

5.1 using SMOTE

In [24]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = (smote.fit_resample(X_train_std,y_train))

In [25]:
#Checking the new balanced distribution
print("Original class distribution:", np.bincount(y_train))
print("Resampled class distribution:", np.bincount(y_train_smote))

Original class distribution: [3614 1308]
Resampled class distribution: [3614 3614]


5.2 using Random Undersampling

In [26]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train_std, y_train)

## **6. Feature selection**

6.1 using Corelation based filtering

In [27]:
corr_df = tele_data.corr(numeric_only=True)

target_corr = (corr_df["Churn"].abs().sort_values(ascending=False))

print(target_corr.head(15))

Churn              1.000000
AvgMonthlySpend    0.423568
tenure             0.354049
ContractValue      0.199675
TotalCharges       0.199484
MonthlyCharges     0.192858
SeniorCitizen      0.150541
ServiceCount       0.086173
Name: Churn, dtype: float64


6.2 using Recursive feature Elimination

In [28]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)

rfe = RFE(estimator=lr,n_features_to_select=15)

rfe.fit(X_train,y_train)

selected_features = (X_train.columns[rfe.support_])

print(selected_features)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Index(['Dependents', 'PhoneService', 'PaperlessBilling',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_Yes', 'OnlineBackup_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service',
       'StreamingMovies_No internet service', 'Contract_One year',
       'Contract_Two year', 'PaymentMethod_Electronic check',
       'PaymentMethod_Mailed check'],
      dtype='object')


6.3 using Mutual information

In [29]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = (mutual_info_classif(X_train,y_train))

mi_df = pd.DataFrame({"Feature":X_train.columns,"MI":mi_scores})

mi_df.sort_values("MI",ascending=False).head(15)

,Feature,MI
9,AvgMonthlySpend,0.132318
4,tenure,0.074088
29,Contract_Two year,0.073918
11,ContractValue,0.050420
7,MonthlyCharges,0.046753
14,InternetService_Fiber optic,0.044423
16,OnlineSecurity_No internet service,0.039465
8,TotalCharges,0.039024
15,InternetService_No,0.036005
18,OnlineBackup_No internet service,0.034859


## **Saving the Dataset**

In [30]:
processed_df = pd.concat([pd.DataFrame(X),pd.DataFrame(y)],axis=1)

processed_df.to_csv("/content/preprocessed-Telco-Customer-Churn_final.csv",index=False)

saving scaler and encoder models

In [31]:
import joblib
import os

output_dir = "models"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

joblib.dump(std_scaler,os.path.join(output_dir, "scaler.pkl"))

joblib.dump(le, os.path.join(output_dir, "label_encoder.pkl"))

joblib.dump(X_train, os.path.join(output_dir, "onehot_encoder.pkl"))

['models/onehot_encoder.pkl']